In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DATABASE = os.getenv("DB_NAME")
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
SCHEMA = os.getenv("DB_SCHEMA")

In [10]:
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
)

In [ ]:
# import pandas as pd
# from pathlib import Path

# query = """
# SELECT *
# FROM uninorte_omop.concept
# """

# concept = pd.read_sql(
#     query,
#     engine
# )

# OUTPUT_DIR = Path(
#     r"C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\data\raw"
# )

# concept.to_csv(
#     OUTPUT_DIR / "concept.csv",
#     index=False
# )

# print(
#     f"Filas: {len(concept):,}"
# )



Filas: 10,004,331


In [3]:
import pandas as pd

query = """
SELECT
    schemaname,
    relname AS tabla,
    n_live_tup AS filas_estimadas
FROM pg_stat_user_tables
WHERE schemaname = 'uninorte_omop'
ORDER BY filas_estimadas DESC;
"""

pd.read_sql(query, engine)

,schemaname,tabla,filas_estimadas
0,uninorte_omop,concept,9646834
1,uninorte_omop,cups,94086
2,uninorte_omop,measurement,0
3,uninorte_omop,procedure_occurrence,0
4,uninorte_omop,person,0
5,uninorte_omop,condition_occurrence,0
6,uninorte_omop,visit_occurrence,0
7,uninorte_omop,visit_detail,0


In [4]:
query = """
SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'uninorte_omop'
AND table_name = 'visit_occurrence'
ORDER BY ordinal_position;
"""

pd.read_sql(query, engine)

,column_name,data_type
0,visit_occurrence_id,integer
1,person_id,integer
2,visit_concept_id,integer
3,visit_start_datetime,timestamp without time zone
4,visit_end_datetime,timestamp without time zone
5,provider_id,integer


In [5]:
tablas = [
    "person",
    "visit_occurrence",
    "visit_detail",
    "condition_occurrence",
    "procedure_occurrence",
    "measurement",
    "concept",
    "cups"
]

In [6]:
import pandas as pd

tablas = [
    "condition_occurrence",
    "procedure_occurrence",
    "measurement"
]

resultados = []

for tabla in tablas:

    query = f"""
    SELECT COUNT(*) AS total
    FROM uninorte_omop.{tabla}
    """

    filas = pd.read_sql(query, engine).iloc[0, 0]

    resultados.append({
        "tabla": tabla,
        "filas": filas
    })

pd.DataFrame(resultados)

,tabla,filas
0,condition_occurrence,38636063
1,procedure_occurrence,47042032
2,measurement,16575888


In [29]:
import os
import pandas as pd

OUTPUT_DIR = r"C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\data\raw"

os.makedirs(OUTPUT_DIR, exist_ok=True)

tablas = [
    "person",
    "visit_occurrence",
    "visit_detail",
    "concept",
    "cups"
]

for tabla in tablas:

    print(f"\nDescargando {tabla}...")

    query = f"""
    SELECT *
    FROM uninorte_omop.{tabla}
    """

    df = pd.read_sql(query, engine)

    print(f"Filas: {len(df):,}")
    print(f"Columnas: {len(df.columns)}")

    ruta = os.path.join(
        OUTPUT_DIR,
        f"{tabla}.csv"
    )

    df.to_csv(
        ruta,
        index=False
    )

    print(f"✅ Guardado en: {ruta}")


Descargando person...
Filas: 912,205
Columnas: 5
✅ Guardado en: C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\data\raw\person.csv

Descargando visit_occurrence...
Filas: 1,706,001
Columnas: 6
✅ Guardado en: C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\data\raw\visit_occurrence.csv

Descargando visit_detail...
Filas: 3,281,613
Columnas: 6
✅ Guardado en: C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\data\raw\visit_detail.csv

Descargando concept...
Filas: 10,004,331
Columnas: 4
✅ Guardado en: C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\data\raw\concept.csv

Descargando cups...
Filas: 94,086
Columnas: 3
✅ Guardado en: C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\data\raw\cups.csv


In [7]:
import os
import pandas as pd

OUTPUT_DIR = r"C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\data\raw"

os.makedirs(OUTPUT_DIR, exist_ok=True)

tablas = [
    "condition_occurrence",
    "procedure_occurrence",
    "measurement"
]

for tabla in tablas:

    print(f"\nDescargando muestra de {tabla}")

    query = f"""
    SELECT *
    FROM uninorte_omop.{tabla}
    TABLESAMPLE SYSTEM (10)
    """

    df = pd.read_sql(query, engine)

    print(f"Filas obtenidas: {len(df):,}")

    if len(df) > 1_000_000:
        df = df.sample(
            n=1_000_000,
            random_state=42
        )

    ruta = os.path.join(
        OUTPUT_DIR,
        f"{tabla}_sample_1M.parquet"
    )

    df.to_parquet(
        ruta,
        index=False
    )

    print(
        f"✅ Guardado {ruta} "
        f"({len(df):,} filas)"
    )


Descargando muestra de condition_occurrence
Filas obtenidas: 3,883,208
✅ Guardado C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\data\raw\condition_occurrence_sample_1M.parquet (1,000,000 filas)

Descargando muestra de procedure_occurrence
Filas obtenidas: 4,675,690
✅ Guardado C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\data\raw\procedure_occurrence_sample_1M.parquet (1,000,000 filas)

Descargando muestra de measurement
Filas obtenidas: 1,658,065
✅ Guardado C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\data\raw\measurement_sample_1M.parquet (1,000,000 filas)


In [ ]:
import pandas as pd
import sweetviz as sv

from pathlib import Path

REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("data/raw/visit_occurrence.csv")

report = sv.analyze(df)

report.show_html(
    "reports/visit_occurrence_report.html"
)

                                             |          | [  0%]   00:00 -> (? left)

Report reports/visit_occurrence_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [8]:
from pathlib import Path

import pandas as pd
import sweetviz as sv

RAW_DIR = Path(
    r"C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\data\raw"
)

REPORT_DIR = Path(
    r"C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\reports"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for file in RAW_DIR.glob("*.csv"):

    print(f"\nProcesando CSV: {file.name}")

    df = pd.read_csv(file)

    print(
        f"Filas: {len(df):,} | "
        f"Columnas: {len(df.columns)}"
    )

    report = sv.analyze(
        df,
        pairwise_analysis="off"
    )

    report.show_html(
        filepath=str(
            REPORT_DIR /
            f"{file.stem}_report.html"
        ),
        open_browser=False
    )

    print(
        f"✅ Reporte generado: "
        f"{file.stem}_report.html"
    )



for file in RAW_DIR.glob("*.parquet"):

    print(f"\nProcesando PARQUET: {file.name}")

    df = pd.read_parquet(file)

    print(
        f"Filas: {len(df):,} | "
        f"Columnas: {len(df.columns)}"
    )

    report = sv.analyze(
        df,
        pairwise_analysis="off"
    )

    report.show_html(
        filepath=str(
            REPORT_DIR /
            f"{file.stem}_report.html"
        ),
        open_browser=False
    )

    print(
        f"✅ Reporte generado: "
        f"{file.stem}_report.html"
    )


Procesando CSV: concept.csv
Filas: 10,004,331 | Columnas: 4


                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\reports\concept_report.html was generated.
✅ Reporte generado: concept_report.html

Procesando CSV: cups.csv
Filas: 94,086 | Columnas: 3


                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\reports\cups_report.html was generated.
✅ Reporte generado: cups_report.html

Procesando CSV: person.csv
Filas: 912,205 | Columnas: 5


...

Report C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\reports\condition_occurrence_sample_1M_report.html was generated.
✅ Reporte generado: condition_occurrence_sample_1M_report.html

Procesando PARQUET: measurement_sample_1M.parquet
Filas: 1,000,000 | Columnas: 7


                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\reports\measurement_sample_1M_report.html was generated.
✅ Reporte generado: measurement_sample_1M_report.html

Procesando PARQUET: procedure_occurrence_sample_1M.parquet
Filas: 1,000,000 | Columnas: 5


                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\taran\OneDrive\Documents\dataChallenge\HUN\patient-flow-HUN\reports\procedure_occurrence_sample_1M_report.html was generated.
✅ Reporte generado: procedure_occurrence_sample_1M_report.html
